# P07 · Análisis de derivados financieros
**QuantLab / Módulo 1 · Fundamentos financieros / Práctica 07 de 12**

**Objetivo:** entender la información adicional de los perpetuos y distinguir una relación estadística de una oportunidad económica. Tiempo orientativo: 90–120 minutos. Requisitos: P01–P06, Pandas y NumPy.

Trabajaremos con un contrato lineal BTC/USDT y datos horarios. La ejecución por defecto utiliza datos **sintéticos**, no cotizaciones reales. Puedes cargar tu histórico normalizado mediante CSV y, de forma independiente, consultar la API pública de OKX. No se envían órdenes ni se necesitan claves.

Al terminar podrás interpretar Basis, interés abierto, financiación, liquidaciones, cobertura y riesgo de margen. Los ejercicios están resueltos en la sección 11.

Las funciones están en **`src/quantlab/p07/derivatives.py`** y sus pruebas en **`tests/test_derivatives.py`** (`pytest` desde la raíz del proyecto).

## 1. Configuración y supuestos
Las tarifas son hipotéticas. Las tasas de financiación se expresan como decimales por pago (0,0001 = 0,01 % = 1 bps). Los precios están en USDT por BTC, el OI en BTC equivalentes y las liquidaciones en nominal USDT. USDT no es EUR ni necesariamente USD.

Un timestamp representa el cierre de una ventana horaria UTC, cuando sus datos se consideran disponibles. Para datos reales usa la hora de disponibilidad efectiva; no basta la fecha económica. No se mezclan contratos inversos ni se reutilizan automáticamente datos P06 en EUR.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
from pathlib import Path
from IPython.display import display
import inspect
import sys

try:
    import quantlab
except ImportError:  # sin `pip install -e .`: buscamos src/ subiendo desde el cuaderno
    src = next((d / "src" for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "quantlab").exists()), None)
    if src is None:
        raise
    sys.path.insert(0, str(src))
from quantlab.p07.derivatives import (COLS, EVENT_LABELS, PRICE_COLS, add_features, apply_price_shock, basis_bps,
                                      carry_decomposition, event_study, event_study_horizons, funding_payments,
                                      hedge_curve, make_synthetic, margin_breach_table, margin_scenario, normalize,
                                      oi_change_comparison, oi_regime, okx_snapshot)

DATA_MODE = "synthetic"  # synthetic | csv
CSV_PATH = Path("p07_derivatives_hourly.csv")
RUN_LIVE = False  # captura pública independiente; no sustituye el histórico
OKX_BASE = "https://www.okx.com"  # usa el dominio indicado por tu región si procede
EXPORT = False
SEED = 72
HORIZON = 3  # horas para el estudio de eventos
FEE_BPS = 10  # por ejecución, hipotético
QTY_BTC = .02
MARGIN_USDT = 600.0
MAINTENANCE_RATE = .005  # supuesto docente, no regla de OKX
plt.rcParams.update({"figure.figsize": (10, 3.3), "axes.spines.top": False,
                     "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .2, "font.size": 11, "figure.dpi": 110})

## 2. Datos y contrato
| Campo | Unidad / significado |
|---|---|
| `timestamp` | UTC, cierre horario y disponibilidad asumida |
| `spot_price`, `perp_price` | último precio observado, USDT/BTC |
| `mark_price`, `index_price` | referencia de valoración y referencia del subyacente |
| `open_interest_btc` | OI convertido a equivalentes BTC; un contrato cuenta una vez |
| `funding_rate` | tasa **realizada** por pago; cero solo cuando sabemos que no hubo pago; NaN = desconocido |
| `long_liquidation_usdt`, `short_liquidation_usdt` | nominal cerrado forzosamente durante la hora; cero confirmado o NaN desconocido |

No se puede inferir una serie de liquidaciones completa a partir de una captura puntual. En CSV el contrato, exchange y unidad deben ser homogéneos. Para el ejercicio suponemos financiación liquidada sobre el mark al cierre de la hora de pago. En datos reales puede ser necesario guardar el mark exacto de liquidación de funding.

In [ ]:
print(inspect.getsource(make_synthetic))

raw = make_synthetic(seed=SEED) if DATA_MODE == "synthetic" else pd.read_csv(CSV_PATH) if DATA_MODE == "csv" else None
if raw is None: raise ValueError("DATA_MODE debe ser synthetic o csv")
print("Modo:", DATA_MODE, "| Filas:", len(raw))
display(raw.head(4))

### 2.1 Validación de la serie
No se rellenan huecos de precios, OI ni liquidaciones. Reindexar permite que `shift(3)` signifique tres horas, no tres filas de distancia arbitraria. Una fila duplicada requiere resolución en origen.

<div style="background-color:rgba(240,140,0,0.14); border-left:6px solid #f08c00; padding:12px 16px; border-radius:6px;">

<b style="color:#f08c00;">🔎 FASE DE VALIDACIÓN · ¿la serie horaria es coherente?</b>

`normalize` se detiene con un error si:

- Faltan columnas, o hay timestamps nulos o duplicados.
- Algún timestamp no es un cierre horario exacto en UTC.
- Hay datos vacíos o infinitos, precios ≤ 0, o OI o liquidaciones negativos.

Después reindexa a una rejilla horaria completa: las horas que faltan aparecen como `NaN`, **sin rellenar**.

</div>

In [ ]:
print(inspect.getsource(normalize))

d = normalize(raw)
print("Cobertura horaria:", len(d), "| huecos de spot:", d.spot_price.isna().sum())
display(d.isna().sum().rename("valores_ausentes").to_frame())

## 3. Basis: precio del perpetuo frente al spot
$$Basis_{bps}=10^4\frac{P_{perp}-P_{spot}}{P_{spot}}$$
Un Basis positivo indica una prima del perpetuo, no una ganancia asegurada. El mark suele usarse para valorar posiciones y liquidaciones; el index es una referencia externa. No son necesariamente precios ejecutables.

In [ ]:
print(inspect.getsource(add_features))

d = add_features(d, QTY_BTC)
print(d[["basis_bps","mark_index_bps"]].describe().round(3))
ax = d[["basis_bps","mark_index_bps"]].plot(color=["#245c86","#b27836"])
ax.axhline(0,color="gray",linewidth=1); ax.set(title=f"Prima del perpetuo y del mark · {DATA_MODE}",ylabel="Puntos básicos",xlabel="UTC")
plt.tight_layout(); plt.show()

## 4. Open Interest: contratos abiertos, no volumen negociado
Un nuevo contrato con una contraparte larga y otra corta aumenta OI en una unidad; cerrar ambas lo reduce. Cambiar de contraparte puede dejarlo constante. En nominal monetario puede cambiar simplemente porque cambia el precio: por eso calculamos la variación del OI en BTC equivalentes. No deducimos cuántos participantes son alcistas.

In [ ]:
ax = d.open_interest_btc.plot(color="#245c86")
ax.set(title=f"Interés abierto · {DATA_MODE}", ylabel="BTC equivalentes", xlabel="UTC")
plt.tight_layout(); plt.show()
display(oi_regime(d))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Basis y OI en la simulación</b>

- **Basis:** media +3,9 bps, entre −17,8 y +27,2 bps. Oscila entre prima y descuento: en la simulación sigue una sinusoide más ruido.
- **Mark frente a index:** media +3,2 bps; el mark se construye con el 80 % de la prima, así que se mueve como el basis, pero más suave.
- **OI frente a precio** (719 horas): 163 horas con precio y OI a la baja, 186 con precio a la baja y OI al alza, 183 con precio al alza y OI a la baja, y 187 con ambos al alza. **Reparto casi uniforme:** en esta serie, la dirección del OI no dice nada sobre la del precio.

</div>

## 5. Funding Rate: pagos entre participantes
En un perpetuo lineal, suponiendo una tasa positiva, los largos pagan y los cortos reciben. Para una posición de $q$ BTC: $$Funding_{short}=q\,P_{mark}\,f$$
La tasa de financiación anunciada no equivale a la finalmente liquidada. No sumes una tasa prevista repetida cada segundo. El intervalo real de pago depende del instrumento.

In [ ]:
paid = d.loc[d.funding_rate.notna() & d.funding_rate.ne(0), "funding_bps"]
ax = paid.plot(kind="line",marker=".",color="#245c86")
ax.axhline(0,color="gray",linewidth=1); ax.set(title=f"Tasas realizadas en pagos no nulos · {DATA_MODE}",ylabel="bps por pago",xlabel="UTC")
plt.tight_layout(); plt.show()
print("Pagos observados no nulos:",len(paid),"| Funding short observado (USDT):",round(d.short_funding_usdt.sum(min_count=1),4))
print("Correlación basis–funding en horas de pago:", round(d.basis_bps.corr(d.funding_bps.where(d.funding_rate.ne(0))), 4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Funding: pequeño, y en la simulación es una función de la prima</b>

- **90 pagos** en 30 días (uno cada 8 horas). Un short de 0,02 BTC habría cobrado **6,14 USDT** en total: unos 0,07 USDT por pago.
- La correlación entre basis y funding en las horas de pago es **1,00**, pero **por construcción**: la simulación define el funding como el 15 % de la prima (con un tope de ±4 bps). En datos reales la relación existe, pero no es exacta ni inmediata.
- Un funding positivo no es un ingreso garantizado: puede cambiar de signo de un pago al siguiente (ver E5 y E7).

</div>

## 6. Liquidaciones: una primera investigación de eventos
$$LI=\frac{L_{long}-L_{short}}{L_{long}+L_{short}}$$
Un LI positivo representa predominio de liquidaciones de largos. No significa que todas las ventas provengan de liquidaciones. Definimos un evento con volumen total superior al percentil 95 **estimado solo en desarrollo** y predominio de largos o cortos. Medimos retorno spot desde el cierre de observación hasta tres horas más tarde. Es asociación descriptiva, no simulación ejecutable: no modelamos latencia ni fills.

Separación temporal 65/35; descartamos objetivos de desarrollo que cruzan la frontera. Los eventos pueden solaparse, por lo que sus resultados no son independientes y no calculamos p-valores ingenuos.

In [ ]:
print(inspect.getsource(event_study))

results, threshold, train, test = event_study(d, HORIZON, .65)
d["forward_return_bps"] = (d.spot_price.shift(-HORIZON)/d.spot_price-1)*10000
print(f"Umbral desarrollo: {threshold:,.0f} USDT | Horizonte: {HORIZON} h")
display(results.round(2))
chart=results.mean_bps.unstack(0).reindex(EVENT_LABELS)
ax=chart.plot(kind="bar",rot=0,color=["#245c86","#b27836"])
ax.axhline(0,color="gray",linewidth=1);ax.set(title=f"Retorno posterior a {HORIZON} h · {DATA_MODE}",ylabel="Retorno medio (bps)",xlabel="")
ax.tick_params(axis="x",labelsize=9);plt.tight_layout();plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Eventos de liquidación: muy pocos y de signo inestable</b>

Umbral (percentil 95 del desarrollo): **73.173 USDT**.

| Periodo | Evento | n | Media a 3 h | Mediana a 3 h |
|---|---|---|---|---|
| Desarrollo | Long dominante | 12 | −20,1 bps | −21,6 bps |
| Desarrollo | Short dominante | 12 | −3,8 bps | −13,5 bps |
| Desarrollo | Resto | 441 | −5,2 bps | −3,0 bps |
| Validación | Long dominante | **4** | **+21,5 bps** | +46,7 bps |
| Validación | Short dominante | 7 | −22,5 bps | −43,4 bps |
| Validación | Resto | 238 | +0,7 bps | −2,2 bps |

- Tras las liquidaciones de largos, el desarrollo sugiere continuación bajista (−20 bps) y la validación, rebote (+21 bps). **El signo se invierte.**
- Con 4 a 12 eventos por grupo, un solo evento mueve la media decenas de bps. No hay base para concluir nada.
- En la simulación, los retornos futuros son independientes del pasado por construcción: estas diferencias son **ruido**.

</div>

## 7. Cobertura spot + short perpetuo
Compramos `q` BTC spot y vendemos en corto `q` BTC de un perpetuo lineal. El PnL combinado por precio es `q*(spot_t-spot_0) - q*(perp_t-perp_0)`. Se cancelan aproximadamente las variaciones comunes de precio, no las variaciones del Basis. Sumamos funding y descontamos cuatro ejecuciones (apertura y cierre de ambas piernas).

La curva supone cierre hipotético en cada hora. No se opera realmente ni se modelan spreads, slippage, impuestos, reequilibrado, financiación del colateral o riesgo de contraparte. El funding de la primera fila se excluye porque entramos después de ese evento. Si faltan pagos o precios, no se interpreta el PnL posterior como conocido. La garantía del derivado es independiente del spot.

In [ ]:
print(inspect.getsource(hedge_curve))

# Una semana de datos: comenzar en hora disponible; no elegir por rendimiento.
segment=d.iloc[:168]
h=hedge_curve(segment,QTY_BTC,MARGIN_USDT,FEE_BPS,MAINTENANCE_RATE)
capital=QTY_BTC*segment.spot_price.iloc[0]+MARGIN_USDT
print("Capital inicial (sin reserva extra de comisiones):",round(capital,2),"USDT")
display(h[["price_pnl","funding_pnl","fees","net_pnl"]].tail(1).round(4))
print("Posible umbral de margen observado:",bool(h.margin_breach.any()),"| PnL posterior no ejecutable si hay liquidación")
ax=h[["price_pnl","funding_pnl","net_pnl"]].plot(color=["#245c86","#b27836","#347c61"])
ax.axhline(0,color="gray",linewidth=1); ax.set(title=f"Cobertura teórica: primera semana · {DATA_MODE}",ylabel="PnL (USDT)",xlabel="UTC")
if h.margin_breach.any(): ax.axvline(h.index[h.margin_breach][0],color="red",linestyle="--",label="Umbral margen"); ax.legend()
plt.tight_layout();plt.show()

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · Cobertura de una semana: el funding no cubre las comisiones</b>

| Componente | USDT |
|---|---|
| Variación del basis (PnL de precio) | −1,54 |
| Funding cobrado (21 pagos) | +3,06 |
| Comisiones (4 ejecuciones, 10 bps) | −4,72 |
| **Neto** | **−3,21** |

- El spot cayó de 59.820 a 58.038 USDT (−3,0 %) y la cobertura apenas lo notó: las dos piernas se compensan.
- Lo que **no** se cancela es el basis: pasó de +0,7 a +14,1 bps, y el short del perpetuo perdió esa diferencia (−1,54 USDT).
- El funding (+3,06) no alcanza para pagar cuatro ejecuciones (−4,72). Resultado: **−0,18 %** sobre un capital de 1.796 USDT.
- No se cruzó el umbral de margen.

</div>

## 8. Apalancamiento y margen: escenario adverso
Un short de nominal 1.000 USDT con garantía 100 USDT tiene apalancamiento inicial x10. Si el precio sube un 5 % pierde aproximadamente 50 USDT (sin costes). Con una tasa de mantenimiento hipotética de 0,5 %, se puede cruzar el umbral antes de perder toda la garantía. No es una calculadora real de liquidación: cada exchange aplica niveles de margen, tarifas, mark y reglas propias; los saltos de precio pueden empeorar las pérdidas.

In [ ]:
scenario = margin_scenario(1000, 100, [0,.02,.05,.08,.10,.15], MAINTENANCE_RATE)
display(scenario.round(2))

## 9. Conexión pública a OKX (opcional)
Documentación: [OKX API v5](https://www.okx.com/docs-v5/en/) y [Kraken Futures Market Analytics](https://docs.kraken.com/api/docs/futures-api/charts/market-analytics). Capturamos snapshots de spot, swap, mark, índice, OI y funding anunciado, además de una muestra del histórico de funding. `oiCcy` expresa OI en moneda base para este instrumento, `oi` en contratos. Los endpoints son secuenciales: los timestamps pueden diferir, no son una única instantánea sincronizada. Una captura no constituye un histórico de OI ni de liquidaciones.

Activa `RUN_LIVE=True` arriba y ejecuta todo. Si el dominio no está disponible en tu región, consulta la documentación regional. No se sustituyen errores por datos ficticios. No se incluyen liquidaciones reales aquí: habrá que consultar una fuente histórica o capturar eventos de un feed documentado, conservando cobertura y unidades.

In [ ]:
print(inspect.getsource(okx_snapshot))

if RUN_LIVE:
    live_rows, live_hist, live = okx_snapshot(OKX_BASE)  # si falla, lanza el error
    display(live_rows)
    live_hist["fundingTime_utc"] = pd.to_datetime(live_hist.fundingTime.astype("int64"), unit="ms", utc=True)
    display(live_hist)
else:
    print("Captura pública desactivada: no se ha llamado a OKX.")

## 10. Comprobaciones automáticas
Se usan casos pequeños conocidos, independientes de los resultados de mercado.

In [ ]:
# 🔎 FASE DE VALIDACIÓN: fórmulas, separación sin fugas, cobertura con casos conocidos y funding desconocido
assert np.isclose((60100/60000-1)*10000, 100/60000*10000)
assert np.isclose(.1*60000*.0001,.6)
assert np.isclose((120-100)/100*100,20)
assert np.isclose((90-10)/(90+10),.8)
assert d.index.is_monotonic_increasing and d.index.is_unique
assert train.index[-1]+pd.Timedelta(hours=HORIZON)<test.index[0]
assert d.liquidation_imbalance.dropna().between(-1,1).all()
assert d.forward_return_bps.tail(HORIZON).isna().all()
case=make_synthetic(3); case.loc[:,PRICE_COLS]=60000; case.loc[:,"funding_rate"]=[.0001,.0001,-.0001]
case_h=hedge_curve(normalize(case),.1,1000,0)
assert np.isclose(case_h.funding_pnl.iloc[0],0)
assert np.isclose(case_h.net_pnl.iloc[1],.6)
assert np.isclose(case_h.net_pnl.iloc[2],0)
case_h_fee=hedge_curve(normalize(case),.1,1000,10)
assert np.isclose(case_h_fee.fees.iloc[-1],24)
case.loc[1,"funding_rate"]=np.nan
assert hedge_curve(normalize(case),.1,1000,0).net_pnl.iloc[1:].isna().all()
print("13 comprobaciones superadas.")
if EXPORT:
    out=Path("p07_output");out.mkdir(exist_ok=True)
    raw.to_csv(out/"derivatives_raw.csv",index=False)
    d.to_csv(out/"derivatives_features.csv")
    import json
    (out/"metadata.json").write_text(json.dumps(dict(mode=DATA_MODE,source="synthetic seed" if DATA_MODE=="synthetic" else str(CSV_PATH),seed=SEED,quote="USDT",frequency="1h",horizon=HORIZON,fee_bps=FEE_BPS),indent=2))

## 11. Ejercicios
**E1. Basis.** Implementa tu función `basis_bps(spot, perp)` sin copiar la implementación y comprueba 60.000 vs 60.300 (50 bps). ¿Qué ocurre si el spot es cero?

**E2. OI.** Compara variación de OI en BTC y USDT. Identifica tres horas donde difieran de signo, si existen. Explica el papel del precio.

**E3. Funding.** Implementa una función para calcular pagos long y short por evento, con tasas negativas y tamaños distintos. Casos de prueba: q=0,1 BTC, mark=60.000, tasa=±0,0001; resultado short ±0,60 USDT.

**E4. Liquidaciones.** Repite el estudio de eventos a 1, 3 y 6 horas. Fija el umbral usando solo desarrollo y registra el número de eventos. No optimices con validación. ¿Son suficientes los datos?

**E5. Carry.** Evalúa comisiones por lado de 0, 10 y 20 bps en la misma semana; separa funding, variación del Basis y comisiones. ¿Cuál es el rendimiento sobre el capital total? ¿Qué pasa si el funding cambia de signo?

**E6. Margen.** Repite la curva con garantías de 100, 300 y 600 USDT. Identifica el primer cruce del umbral, si lo hubiera. Explica por qué el beneficio spot no evita necesariamente liquidar el short.

**E7. Datos reales.** Activa la consulta a OKX y documenta timestamps, unidades, diferencia entre funding anunciado y realizado, y qué campos necesitarías capturar para crear el histórico. Si la API falla, anota el error y consulta documentación regional.

**E8 (ampliación).** Diseña una hipótesis que combine liquidaciones, OI y OFI de P02/P03. Define el instante de disponibilidad, horizonte, referencia y costes. No necesitas operar ni entrenar un modelo.

### E1 · Basis

In [ ]:
print(inspect.getsource(basis_bps))

print("60.000 → 60.300:", basis_bps(60000, 60300), "bps")
print("60.000 → 59.700:", basis_bps(60000, 59700), "bps")
for bad_spot in [0, -1, np.nan]:
    try:
        basis_bps(bad_spot, 60300)
    except ValueError as exc:
        print(f"spot = {bad_spot}: ValueError('{exc}')")
assert np.isclose(basis_bps(60000, 60300), 50)

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E1 · Basis</b>

- 10.000 × (60.300 − 60.000) / 60.000 = **+50 bps**: el perpetuo cotiza con una prima del 0,5 %. Con 59.700, el basis es **−50 bps** (descuento).
- **Spot = 0:** la fórmula divide entre cero. En Python daría un error o infinito; con pandas, `inf` o `NaN` en silencio. Además, no tiene sentido económico: un precio spot de 0 indica un dato corrupto, no un activo gratis. Por eso la función **se detiene** con `ValueError` ante un spot ≤ 0 o no finito, igual que `normalize` rechaza precios no positivos.

</div>

### E2 · OI en BTC frente a USDT

In [ ]:
print(inspect.getsource(oi_change_comparison))

e2 = oi_change_comparison(d)
print(f"Horas con signo distinto: {int(e2.sign_differs.sum())} de {len(e2)}")
display(e2[e2.sign_differs].head(3).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E2 · El precio puede invertir el signo del OI nominal</b>

**235 de 719 horas** (33 %) tienen signo distinto en BTC y en USDT. Tres ejemplos:

| Hora UTC | OI en BTC | OI en USDT | Mark |
|---|---|---|---|
| 01-01 02:00 | −0,016 % | **+0,067 %** | +0,083 % |
| 01-01 06:00 | +0,071 % | **−0,222 %** | −0,294 % |
| 01-01 08:00 | −0,239 % | **+0,087 %** | +0,327 % |

- OI en USDT = OI en BTC × mark. Su variación es aproximadamente **variación del OI en BTC + variación del precio**.
- Cuando el precio se mueve más que el OI y en sentido contrario, el OI nominal cambia de signo **sin que se haya abierto ni cerrado un solo contrato más**. A las 06:00 se abrieron contratos (+0,07 % en BTC), pero el OI en USDT bajó un 0,22 % porque el precio cayó un 0,29 %.
- Por eso el cambio de posicionamiento se mide en **BTC**: el nominal mezcla las posiciones con el precio.

</div>

### E3 · Pagos de funding

In [ ]:
print(inspect.getsource(funding_payments))

e3 = pd.DataFrame([{"q_btc": q, "tasa": rate, **funding_payments(q, 60000, rate)}
                   for q, rate in [(.1, .0001), (.1, -.0001), (.5, .0001), (.02, .00025)]])
display(e3)
assert np.isclose(funding_payments(.1, 60000, .0001)["short_usdt"], .6)
assert np.isclose(funding_payments(.1, 60000, -.0001)["short_usdt"], -.6)

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E3 · Quien está del lado de la prima, paga</b>

| q | Tasa | Long | Short |
|---|---|---|---|
| 0,1 BTC | +0,01 % | −0,60 USDT | **+0,60 USDT** |
| 0,1 BTC | −0,01 % | +0,60 USDT | **−0,60 USDT** |
| 0,5 BTC | +0,01 % | −3,00 USDT | +3,00 USDT |
| 0,02 BTC | +0,025 % | −0,30 USDT | +0,30 USDT |

- Pago = q × mark × tasa. Con tasa **positiva**, los largos pagan y los cortos cobran; con tasa **negativa**, al revés.
- El pago es proporcional al tamaño y a la tasa: es un intercambio entre participantes (lo que paga uno lo cobra el otro), no una comisión del exchange.

</div>

### E4 · Eventos de liquidación a 1, 3 y 6 horas

In [ ]:
print(inspect.getsource(event_study_horizons))

e4 = event_study_horizons(d, (1, 3, 6), .65)
display(e4.round(2))
display(e4["mean_bps"].unstack("periodo").round(2))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E4 · Con 4 a 12 eventos no hay evidencia</b>

Retorno medio posterior (bps). El umbral se fija en el desarrollo de cada horizonte (≈ 73.100–73.300 USDT):

| Horizonte | Long dominante: desarrollo (12) | Validación (4) | Short dominante: desarrollo (12) | Validación (7) |
|---|---|---|---|---|
| 1 h | −8,5 | **+20,2** | +0,9 | −20,0 |
| 3 h | −20,1 | **+21,5** | −3,8 | −22,5 |
| 6 h | −30,7 | −0,6 | +14,6 | −22,1 |

- El número de eventos es el mismo en los tres horizontes: **12 y 12 en desarrollo, 4 y 7 en validación**.
- El signo se invierte entre periodos en 4 de las 6 combinaciones, y media y mediana discrepan mucho (a 6 h, en validación: media −0,6 y mediana +83,1 bps), señal de que unos pocos eventos dominan.
- **Los datos no son suficientes.** Haría falta un histórico de muchos meses, con cientos de eventos, eventos que no se solapen y una validación ajustada por dependencia. No se ha elegido ningún horizonte mirando la validación.

</div>

### E5 · Carry: comisiones, funding y basis

In [ ]:
print(inspect.getsource(carry_decomposition))

e5 = carry_decomposition(segment, QTY_BTC, MARGIN_USDT, (0, 10, 20), MAINTENANCE_RATE)
e5_flip = carry_decomposition(segment, QTY_BTC, MARGIN_USDT, (0, 10, 20), MAINTENANCE_RATE, flip_funding=True)
display(pd.concat({"Funding real": e5, "Funding con signo invertido": e5_flip}).round(4))

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E5 · El carry vive o muere por las comisiones y el signo del funding</b>

Misma semana, 0,02 BTC, capital total 1.796 USDT (spot + garantía):

| Comisión por lado | Basis | Funding | Comisiones | Neto | Sobre el capital |
|---|---|---|---|---|---|
| 0 bps | −1,54 | +3,06 | 0,00 | **+1,51** | +0,08 % |
| 10 bps | −1,54 | +3,06 | −4,72 | −3,21 | −0,18 % |
| 20 bps | −1,54 | +3,06 | −9,43 | −7,92 | −0,44 % |

**Con el funding de signo invertido:** −4,60, −9,32 y −14,03 USDT (de −0,26 % a −0,78 %).

- Solo hay beneficio **sin comisiones**, y es de +0,08 % en una semana sobre todo el capital.
- La rentabilidad debe medirse sobre el capital **total**: hay que comprar el spot (1.196 USDT) **y** depositar la garantía (600 USDT). Medir sobre la garantía sola multiplicaría el resultado por tres.
- **Si el funding cambia de signo,** el short pasa a pagar y se pierde en todos los escenarios. El funding no es un cupón fijo: cambia en cada periodo (en OKX, 5 de los últimos 10 pagos fueron negativos, ver E7).
- El basis aporta riesgo propio: aquí se amplió y costó 1,54 USDT. La cobertura neutraliza la dirección del precio, no el basis.

</div>

### E6 · Margen con garantías de 100, 300 y 600 USDT

In [ ]:
print(inspect.getsource(margin_breach_table))

e6 = margin_breach_table(segment, QTY_BTC, (100, 300, 600), FEE_BPS, MAINTENANCE_RATE)
print("Semana real de la simulación:")
display(e6.round({c: 3 for c in e6.columns if c != "first_breach_utc"}))

# Escenario de estrés: salto del +15 % en los cuatro precios desde la hora 100 de la semana
stressed = apply_price_shock(segment, segment.index[100], .15)
e6_stress = margin_breach_table(stressed, QTY_BTC, (100, 300, 600), FEE_BPS, MAINTENANCE_RATE)
print("Con un salto de precio del +15 % en", segment.index[100])
display(e6_stress.round({c: 3 for c in e6_stress.columns if c != "first_breach_utc"}))
hs = hedge_curve(stressed, QTY_BTC, 100, FEE_BPS, MAINTENANCE_RATE)
first = hs.index[hs.margin_breach][0]
print(f"Garantía 100 USDT en {first}: patrimonio del margen {hs.margin_equity.loc[first]:.2f} USDT "
      f"| mantenimiento {QTY_BTC*hs.mark_price.loc[first]*MAINTENANCE_RATE:.2f} | PnL total de la cobertura {hs.net_pnl.loc[first]:.2f}")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E6 · El spot gana, pero el margen del short está en otra cuenta</b>

**Semana real de la simulación:** ninguna garantía cruza el umbral. El mark llegó a subir solo un 1,1 %, y la mínima garantía restante con 100 USDT fue de 87,2 USDT.

| Garantía | Apalancamiento inicial | Subida del mark que cruzaría el umbral |
|---|---|---|
| 100 USDT | x12,0 | **+7,7 %** |
| 300 USDT | x4,0 | +24,4 % |
| 600 USDT | x2,0 | +49,3 % |

**Escenario de estrés** (salto del +15 % en los cuatro precios el 5 de enero a las 05:00): con **100 USDT** se cruza el umbral **en la misma hora del salto**. El mark está un 11,3 % por encima de la entrada, el patrimonio de la cuenta de margen es de −34,5 USDT (mantenimiento: 6,7) y el umbral sigue superado durante 45 horas. Con 300 y 600 USDT no se cruza.

**¿Por qué el beneficio del spot no lo evita?** En ese instante la pierna spot gana **+135,6 USDT** y la cobertura completa va casi plana (**−2,6 USDT**). Pero el BTC comprado está en la cuenta spot (o en otra plataforma) y la garantía del short es una cuenta **aislada**: el exchange solo mira esa cuenta. Si no se aporta más garantía a tiempo, liquida el short, y la posición queda **larga y sin cobertura**, justo después de un salto. La neutralidad direccional del conjunto no protege a cada pierna por separado.

</div>

### E7 · Datos reales de OKX
Cambia `RUN_LIVE_E7` a `True` para capturar la instantánea pública. Si la API falla, la celda se detiene con el error.

In [ ]:
RUN_LIVE_E7 = False

if RUN_LIVE_E7 or RUN_LIVE:
    e7_rows, e7_hist, e7_live = okx_snapshot(OKX_BASE)
    display(e7_rows)
    e7_hist["fundingTime_utc"] = pd.to_datetime(e7_hist.fundingTime.astype("int64"), unit="ms", utc=True)
    display(e7_hist)
    values = {(r.source, r.field): r.value for r in e7_rows.itertuples()}
    spot, perp = float(values[("spot", "last")]), float(values[("perp", "last")])
    print(f"Basis: {basis_bps(spot, perp):.2f} bps | OI: {float(values[('oi','oi')]):,.0f} contratos × "
          f"{values[('instrument','ctVal')]} BTC = {float(values[('oi','oiCcy')]):,.2f} BTC")
    print("Funding anunciado:", values[("funding", "fundingRate")], "| último liquidado:", values[("funding", "settFundingRate")])
else:
    print("RUN_LIVE_E7 = False: no se consulta OKX. Resultados documentados en la conclusión de abajo.")

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E7 · OKX BTC-USDT-SWAP (19-11-2026, 20:33:55–20:33:58 UTC)</b>

| Campo | Valor | Unidad | `ts` del exchange (UTC) |
|---|---|---|---|
| Spot (`last`, BTC-USDT) | 61.968,0 | USDT/BTC | 20:33:54,373 |
| Perpetuo (`last`) | 61.936,5 | USDT/BTC | 20:33:54,570 |
| Mark (`markPx`) | 61.953,4 | USDT/BTC | 20:33:55,770 |
| Índice (`idxPx`) | 61.990,2 | USDT/BTC | 20:33:55,557 |
| OI (`oi`) | 3.306.060 | **contratos** | 20:33:56,731 |
| OI (`oiCcy`) | 33.060,6 | **BTC** | 20:33:56,731 |
| Funding anunciado (`fundingRate`) | +0,0000693 (+0,69 bps) | por pago | 20:33:23,499 |
| Último funding liquidado (`settFundingRate`) | −0,0000179 (−0,18 bps) | por pago | — |
| Valor del contrato (`ctVal`) | 0,01 | BTC/contrato | — |

- **Basis: −5,1 bps** (el perpetuo cotizaba con descuento); mark frente a índice, −5,9 bps.
- **Unidades del OI:** 3.306.060 contratos × 0,01 BTC = 33.060,6 BTC, que coincide con `oiCcy`. Sin el `ctVal` del instrumento, el número de contratos no se puede convertir.
- **Timestamps:** la captura es secuencial. Las recepciones abarcan 3,4 s, los `ts` de precios 1,4 s (2,4 s si se incluye el del OI), y el del funding es 33 s más antiguo que el del OI. **No es una instantánea sincronizada.**
- **Anunciado frente a realizado:** la tasa anunciada para el pago de las 00:00 UTC era **+0,69 bps**, pero la última liquidada (16:00 UTC) fue **−0,18 bps**: tienen **signo contrario**. Usar la anunciada como si fuera la cobrada habría invertido el sentido del pago. En el histórico, `fundingRate` coincide con `realizedRate` porque ya está liquidado.
- **Histórico de funding (10 pagos, cada 8 h, del 16 al 19 de noviembre):** 5 positivos y 5 negativos, entre −0,18 y +0,63 bps.

**Campos que habría que capturar para un histórico horario:** hora de recepción local y `ts` del exchange de cada endpoint; spot, perpetuo, mark e índice; `oi` y `oiCcy` junto con `ctVal` (por si cambia); el funding **liquidado** de cada pago (`fundingTime`, `realizedRate`) y, por separado, la tasa anunciada con su hora de captura; el mark en el instante exacto del pago; y las liquidaciones de un feed documentado (WebSocket de liquidaciones), con su cobertura y sus huecos marcados.

</div>

### E8 · Hipótesis: liquidaciones, OI y OFI

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIÓN · E8 · Hipótesis de investigación</b>

**Hipótesis (H1).** En el perpetuo BTC-USDT-SWAP de OKX, cuando en una ventana de 1 minuto coinciden (a) liquidaciones de largos por encima del percentil 99 del desarrollo, (b) una caída del OI en BTC (cierre forzoso de posiciones, no apertura de cortos) y (c) un OFI del libro (P02) que vuelve a positivo en los 30 s siguientes, el retorno del **mid** en los 5 minutos posteriores es positivo (rebote tras la liquidación en cascada).

**H0 (a rechazar):** el retorno medio del mid tras esas ventanas es ≤ el de las ventanas con liquidaciones grandes que **no** cumplen (b) y (c).

**Instante de disponibilidad:**
- Liquidaciones: hora de recepción del mensaje del WebSocket de liquidaciones, no la hora del evento.
- OI: el último valor recibido antes del cierre de la ventana (el endpoint se actualiza con retraso; se guarda su `ts` y la recepción).
- OFI: calculado con los snapshots del libro recibidos dentro de la ventana, por número de secuencia.
- La señal existe al **cierre de la ventana + latencia de recepción**; la entrada de referencia es el mid 1 s después.

**Horizonte:** 5 minutos desde la entrada de referencia. Las ventanas que se solapen con un evento anterior aún abierto se descartan.

**Referencia (benchmark):** las ventanas con liquidaciones de largos grandes que no cumplen las condiciones de OI y OFI, y el retorno incondicional de 5 minutos en la misma hora del día.

**Costes:** spread del perpetuo en el instante de entrada y salida (P01), slippage recorriendo el libro para el tamaño elegido (P02), comisión taker por lado (P08) y funding si la ventana cruza un pago. La diferencia debe superar el coste de ida y vuelta, no solo ser positiva.

**Validación:** varios meses de captura continua; desarrollo en los primeros meses para fijar percentiles y ventanas, y una única validación en los últimos. Intervalos de confianza por bloques, por la dependencia entre eventos cercanos.

</div>

## 12. Conclusiones y siguiente paso

<div style="background-color:rgba(47,158,68,0.14); border-left:6px solid #2f9e44; padding:12px 16px; border-radius:6px;">

<b style="color:#2f9e44;">✅ CONCLUSIONES DE LA PRÁCTICA P07</b>

**¿Por qué OI no mide el número de compradores frente al de vendedores?**
Porque cada contrato abierto tiene siempre un largo **y** un corto: el OI cuenta contratos, no personas ni direcciones. Si sube, se abrieron contratos nuevos, pero no dice de qué lado vino la iniciativa. Además, en USDT el OI cambia con el precio aunque no se abra ninguna posición (235 de 719 horas cambiaron de signo al pasar a USDT).

**¿Qué condiciones deben cumplirse para que una estrategia de funding tenga rentabilidad neta?**
Que el funding cobrado supere, durante el tiempo que se mantiene la posición, las cuatro comisiones de apertura y cierre, el spread y el slippage, la variación adversa del basis y el coste del capital inmovilizado (spot + garantía). Además, el funding debe mantener su signo: en la semana simulada solo hubo beneficio con comisión 0 (+0,08 %), y con el signo invertido se perdía en todos los casos.

**¿Qué limitaciones tienen las liquidaciones disponibles y el estudio de eventos?**
Las APIs públicas no dan un histórico completo de liquidaciones: hace falta capturarlas de un feed, con su cobertura. El estudio tuvo 4 a 12 eventos por grupo, eventos que pueden solaparse, signos que se invierten entre periodos y ninguna prueba ajustada por dependencia. En la simulación, además, los retornos futuros son independientes por construcción.

**¿Qué diferencia existe entre neutralidad direccional y ausencia de riesgo?**
La cobertura spot + short neutraliza la **dirección** del precio, pero no el riesgo de basis, el de cambio de signo del funding, el de margen (E6: con un salto del 15 %, la cuenta del short se liquida aunque el conjunto vaya plano), el de contraparte del exchange, el de ejecución ni el de que USDT pierda su paridad.

**Criterio de finalización:** E1–E7 resueltos, unidades explicadas (USDT/BTC, BTC, contratos, tasa por pago, bps) y ningún cálculo usa datos futuros: el umbral de eventos se fija en el desarrollo, la separación descarta objetivos que cruzan la frontera y la cobertura excluye el funding anterior a la entrada. Los datos sintéticos prueban el procedimiento, no la existencia de Alpha.

📄 Resumen en `docs/conclusiones/P07_derivatives.md`

</div>

**P08:** comisiones y costes operativos: spread, slippage, ejecución maker/taker y punto de equilibrio.